## ESPnet Text-to-Speech Demo


In [1]:
from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from huggingface_hub import snapshot_download
from pathlib import Path

def download_vocoder(tag: str) -> str:
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)

    for file in repo_path.rglob("*.pkl"):
        print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError("No .pkl file found in the vocoder repo")
   

def init_tts(am_file=None, am_tag=None, vocoder_file=None, vocoder_tag=None):
    voc_str = vocoder_tag
    if not voc_str:
        voc_str = vocoder_file
    if not voc_str:
        voc_str = "Griffin-Lim"
    am_str = am_tag
    if not am_str:
        am_str = am_file    

    print(f"\n===================================\nAM      = {am_str}")
    print(f"Vocoder = {voc_str}")
    print(f"===================================")
    
    if vocoder_tag and not vocoder_tag.startswith("parallel_wavegan"):
        vocoder_file = download_vocoder(vocoder_tag)
        vocoder_tag = None
    
    tts = Text2Speech.from_pretrained(
        model_file=am_file,
        model_tag=am_tag,
        vocoder_file=vocoder_file,  # use if local
        vocoder_tag=vocoder_tag,
        device="cpu"
    )
    return tts

print ("ESPnet initialized")    

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'
ESPnet initialized


### Select model

In [4]:
am_tag_hf="Airenas/vdu-arn.fastspeech2.v01"

vocoder_file_600k= "voc/exp/train_nodev_ljspeech_style_melgan.v1/checkpoint-600000steps.pkl"
vocoder_file_700k= "voc/exp/train_nodev_ljspeech_style_melgan.v1/checkpoint-1050000steps.pkl"
vocoder_file_800k= "voc/exp/train_nodev_ljspeech_style_melgan.v1/checkpoint-1000000steps.pkl"
vocoder_file_940k= "voc/exp/train_nodev_ljspeech_style_melgan.v1/checkpoint-1080000steps.pkl"

tts_1 = init_tts(am_tag=am_tag_hf, vocoder_file=vocoder_file_600k)
tts_2= init_tts(am_tag=am_tag_hf, vocoder_file=vocoder_file_700k)
tts_3 = init_tts(am_tag=am_tag_hf, vocoder_file=vocoder_file_800k)
tts_4 = init_tts(am_tag=am_tag_hf, vocoder_file=vocoder_file_940k)

tts_list = [tts_1, tts_2, tts_3, tts_4]

print (f"\n\nREADY: {len(tts_list)} models loaded\n")  




AM      = Airenas/vdu-arn.fastspeech2.v01
Vocoder = voc/exp/train_nodev_ljspeech_style_melgan.v1/checkpoint-600000steps.pkl


Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

/home/airenas/miniconda3/envs/espnet/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



AM      = Airenas/vdu-arn.fastspeech2.v01
Vocoder = voc/exp/train_nodev_ljspeech_style_melgan.v1/checkpoint-1050000steps.pkl


Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

...

/home/airenas/miniconda3/envs/espnet/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



AM      = Airenas/vdu-arn.fastspeech2.v01
Vocoder = voc/exp/train_nodev_ljspeech_style_melgan.v1/checkpoint-1080000steps.pkl


Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]



READY: 4 models loaded



/home/airenas/miniconda3/envs/espnet/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


### Enter text to read

In [5]:
### Kai kurie tekstai paimti iš lrt.lt
texts = ["Sveiki, aš naujas lietuviškas balsas.", 
         "Aš esu labai gerai įrašytas.", 
         "Kaip Jums patinku?", 
         "Pernai maitinimo sektorių sukrėtė dešimtmečius veikusių restoranų ir kavinių bankrotai.", 
         "Šiomis dienomis orai Lietuvoje ims šilti, kris šlapdriba, sniegas, reikės pasisaugoti lijundros.", 
         "Tai savo „Facebook“ paskyroje sekmadienį vakare pranešė Ukrainos pirmasis vicepremjeras ir energetikos ministras Denysas Šmyhalis, skelbia „Ukrinform“."
        ]
for i, text in enumerate(texts):
    print(f"\n==========================================================================\nText = {text}")
    for it, tts in enumerate(tts_list):
        wav = tts(text)["wav"]
        filename = f"output_{i}_{it}.wav"
        sf.write(filename, wav.numpy(), tts.fs)
        audio_widget = Audio(filename)._repr_html_()
        display(HTML(f"<div style='display:flex; align-items:center; gap:10px;'><span>model {it}:</span>{audio_widget}</div>"))



Text = Sveiki, aš naujas lietuviškas balsas.



Text = Aš esu labai gerai įrašytas.



Text = Kaip Jums patinku?



Text = Pernai maitinimo sektorių sukrėtė dešimtmečius veikusių restoranų ir kavinių bankrotai.



Text = Šiomis dienomis orai Lietuvoje ims šilti, kris šlapdriba, sniegas, reikės pasisaugoti lijundros.



Text = Tai savo „Facebook“ paskyroje sekmadienį vakare pranešė Ukrainos pirmasis vicepremjeras ir energetikos ministras Denysas Šmyhalis, skelbia „Ukrinform“.
